# Deep Search with the Sandbox Tool

Using the `web_search` tool in the Agent API returns ranked links and snippets, and reasons over them. That's plenty when the answer
is written in prose -- but some facts only live in structured metadata or deep in a long document,
where a snippet doesn't have it. The `sandbox` [tool](https://docs.perplexity.ai/docs/agent-api/tools/sandbox) gives the model another move: write Python that
drives the same web search and page fetches *as code*, then parses the result. That's
[Search as Code](https://research.perplexity.ai/articles/rethinking-search-as-code-generation).

We test both on two questions -- one whose answer **is** in snippets, one whose answer **isn't** —- each
with `web_search` alone vs. `web_search + sandbox`. 

## 1. Setup

Everything goes through the official [Perplexity Python SDK](https://github.com/ppl-ai/perplexity-python).
We submit jobs with `background=True` and poll `responses.retrieve` — the `sandbox` tool only runs on
the background path.

> Set your key as `PERPLEXITY_API_KEY` (or `PPLX_API_KEY`) — get one at
> [perplexity.ai/account/api](https://www.perplexity.ai/account/api/keys).

In [1]:
%pip install --quiet perplexityai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

You can choose any frontier model from here: https://docs.perplexity.ai/docs/agent-api/models

In [2]:
import os, time, json
from perplexity import Perplexity

API_KEY = os.environ.get("PERPLEXITY_API_KEY") or os.environ.get("PPLX_API_KEY", "pplx-YOUR-KEY-HERE")
MODEL = "openai/gpt-5.5"

assert API_KEY.startswith("pplx-"), "Set PERPLEXITY_API_KEY (or PPLX_API_KEY) to your Perplexity API key."

client = Perplexity(api_key=API_KEY)
print("Perplexity SDK ready | model:", MODEL)

Perplexity SDK ready | model: openai/gpt-5.5


## 2. The A/B harness

`run(prompt, use_sandbox)` creates a background job and polls until it's done. The only thing that
changes between runs is the tool list -- `web_search`, optionally plus `sandbox`. Same model, prompt,
and `max_steps`, so the comparison is fair. `poll` returns the response as a dict so the helpers below
can read it.

In [3]:
def submit(prompt: str, use_sandbox: bool, max_steps: int = 20) -> str:
    """Create a background Agent API job via the SDK. Returns the response id."""
    tools = [{"type": "web_search"}]
    if use_sandbox:
        tools.append({"type": "sandbox"})
    resp = client.responses.create(
        model=MODEL, input=prompt, background=True, max_steps=max_steps, tools=tools,
    )
    return resp.id

def poll(resp_id: str, interval: int = 12, timeout_s: int = 1200) -> dict:
    """Poll a background job until it reaches a terminal state, tolerating transient errors."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            resp = client.responses.retrieve(resp_id)
        except Exception as e:
            print(f"  transient error, retrying: {e}")
            time.sleep(interval)
            continue
        data = resp.model_dump(warnings=False)
        status = data.get("status")
        print(f"  {resp_id[:18]}… -> {status}")
        if status in ("completed", "failed", "incomplete"):
            return data
        time.sleep(interval)
    raise TimeoutError(f"{resp_id} did not finish within {timeout_s}s")

def run(prompt: str, use_sandbox: bool, **kw) -> dict:
    label = "web_search + sandbox" if use_sandbox else "web_search only"
    print(f"Submitting [{label}] …")
    return poll(submit(prompt, use_sandbox, **kw))

### Reading the response

The response has typed `output` items. We use three: `message` (the answer), `sandbox_results` (each
Python cell the model ran), and `usage` (tokens and cost).

In [4]:
def answer_text(resp: dict) -> str:
    for o in resp.get("output", []):
        if o.get("type") == "message":
            return "".join(c.get("text", "") for c in o.get("content", []))
    return ""

def usage_summary(resp: dict) -> dict:
    u = resp.get("usage", {}) or {}
    cost = (u.get("cost") or {}).get("total_cost")
    return {"input_tokens": u.get("input_tokens"), "output_tokens": u.get("output_tokens"),
            "total_cost_usd": cost}

def sandbox_cells(resp: dict) -> list:
    cells = []
    for o in resp.get("output", []):
        if o.get("type") == "sandbox_results":
            res = (o.get("results") or [{}])[0]
            cells.append({"code": o.get("code", ""), "exit_code": res.get("exit_code"),
                          "duration_ms": res.get("duration_ms"), "stdout": res.get("stdout", ""),
                          "stderr": res.get("stderr", "")})
    return cells

### Scorecard and trace helpers

`print_scorecard` lays two runs side by side; `print_trace` dumps the Python the sandbox actually ran.

In [5]:
def make_row(label, resp, score, max_score):
    u = usage_summary(resp)
    return {"run": label, "correct": f"{score}/{max_score}", "input_tokens": u["input_tokens"],
            "output_tokens": u["output_tokens"], "total_cost_usd": u["total_cost_usd"],
            "sandbox_cells": len(sandbox_cells(resp))}

def print_scorecard(rows):
    cols = ["run", "correct", "input_tokens", "output_tokens", "total_cost_usd", "sandbox_cells"]
    w = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
    print(" | ".join(c.ljust(w[c]) for c in cols))
    print("-+-".join("-" * w[c] for c in cols))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(w[c]) for c in cols))

def print_trace(resp, max_chars=900):
    cells = sandbox_cells(resp)
    print(f"The model ran {len(cells)} sandbox cell(s).\n")
    for i, c in enumerate(cells, 1):
        print("=" * 72)
        print(f"CELL {i}   (exit={c['exit_code']}, {c['duration_ms']} ms)")
        print("=" * 72)
        print(c["code"])
        if c["stdout"]:
            print("--- stdout ---"); print(c["stdout"][:max_chars])
        if c["stderr"]:
            print("--- stderr ---"); print(c["stderr"][:400])
        print()

## Example 1: Fetching npm latest versions: `web_search` vs `sandbox`

**Question:** the current published version of 15 npm packages, sourced from the web with a citation
for each.

Both runs search the web -- the difference is *how*. `web_search` pulls snippets into the model's
context and reasons over them. The sandbox uses the same search stack **as code** — `pplx_sdk.search.web`
to find the official source, `content.fetch` to read it -- parses the version, and returns just that.
"Latest version" isn't in a snippet, so the first approach has to hunt; the second reads it straight
off the page. We grade both against the npm registry, so the answer key can't go stale.

In [6]:
NPM_PACKAGES = ["react", "lodash", "express", "axios", "webpack", "typescript", "eslint",
                "next", "vue", "chalk", "commander", "jest", "vite", "redux", "zod"]

NPM_PROMPT = (
    "Build a reference table of the current latest published version of these npm packages. "
    "Source every value from the web and cite the exact URL you read it from; use the web-search "
    "and page-fetch tools available to you rather than any hardcoded API client, and don't answer "
    "from prior knowledge. Return one Markdown row per package: package | latest version | source URL. "
    "Packages: " + ", ".join(NPM_PACKAGES) + "."
)
print(NPM_PROMPT)

Build a reference table of the current latest published version of these npm packages. Source every value from the web and cite the exact URL you read it from; use the web-search and page-fetch tools available to you rather than any hardcoded API client, and don't answer from prior knowledge. Return one Markdown row per package: package | latest version | source URL. Packages: react, lodash, express, axios, webpack, typescript, eslint, next, vue, chalk, commander, jest, vite, redux, zod.


### Ground truth from the registry

The npm registry is authoritative, so we use it as the answer key -- one small request per package for
its `latest` version. (This is just the grader; the model isn't allowed to shortcut to it.)

In [7]:
import urllib.request  # answer key only — an independent source of truth for grading

def npm_latest(pkg):
    with urllib.request.urlopen(f"https://registry.npmjs.org/{pkg}/latest", timeout=20) as r:
        return json.load(r)["version"]

NPM_TRUTH = {pkg: npm_latest(pkg) for pkg in NPM_PACKAGES}

def grade_versions(resp, truth):
    """Count how many true version strings appear verbatim in the answer."""
    text = answer_text(resp)
    hits = {pkg: (ver in text) for pkg, ver in truth.items()}
    return sum(hits.values()), len(hits)

print(NPM_TRUTH)

{'react': '19.2.7', 'lodash': '4.18.1', 'express': '5.2.1', 'axios': '1.17.0', 'webpack': '5.107.2', 'typescript': '6.0.3', 'eslint': '10.4.1', 'next': '16.2.9', 'vue': '3.5.35', 'chalk': '5.6.2', 'commander': '15.0.0', 'jest': '30.4.2', 'vite': '8.0.16', 'redux': '5.0.1', 'zod': '4.4.3'}


### Run both ways

In [8]:
npm_baseline = run(NPM_PROMPT, use_sandbox=False)
npm_sandbox  = run(NPM_PROMPT, use_sandbox=True)

print("\nbaseline answer:\n", answer_text(npm_baseline))
print("\nsandbox answer:\n", answer_text(npm_sandbox))

Submitting [web_search only] …


  resp_42e849da-ad20… -> queued


  resp_42e849da-ad20… -> in_progress


  resp_42e849da-ad20… -> in_progress


  resp_42e849da-ad20… -> in_progress


...

  resp_3aeb283f-07df… -> in_progress


  resp_3aeb283f-07df… -> in_progress


  resp_3aeb283f-07df… -> in_progress


  resp_3aeb283f-07df… -> in_progress


  resp_3aeb283f-07df… -> completed

baseline answer:
 | package | latest version | source URL |
|---|---:|---|
| react | 19.2.7 | https://security.snyk.io/package/npm/react |
| lodash | 4.18.1 | https://security.snyk.io/package/npm/lodash |
| express | 5.2.1 | https://security.snyk.io/package/npm/express |
| axios | 1.17.0 | https://security.snyk.io/package/npm/axios |
| webpack | 5.105.4 | https://security.snyk.io/package/npm/webpack |
| typescript | 6.0.3 | https://security.snyk.io/package/npm/typescript |
| eslint | 10.4.1 | https://secure.software/npm/packages/eslint/10.4.1 |
| next | 16.2.7 | https://security.snyk.io/package/npm/next |
| vue | 3.5.35 | https://snyk.io/test/npm/vue@3.5.35 |
| chalk | 5.6.2 | https://security.snyk.io/package/npm/chalk |
| commander | 14.0.3 | https://security.snyk.io/package/npm/commander |
| jest | 30.4.2 | https://security.snyk.io/package/npm/jest |
| vite | 8.0.16 | https://secure.software/npm/packages/vite/8.0.16 |
| redux | 5.0.1 | https://libu

### Scorecard

In [9]:
print_scorecard([
    make_row("web_search only",      npm_baseline, *grade_versions(npm_baseline, NPM_TRUTH)),
    make_row("web_search + sandbox", npm_sandbox,  *grade_versions(npm_sandbox,  NPM_TRUTH)),
])

run                  | correct | input_tokens | output_tokens | total_cost_usd | sandbox_cells
---------------------+---------+--------------+---------------+----------------+--------------
web_search only      | 12/15   | 597642       | 3251          | 1.01768        | 0            
web_search + sandbox | 15/15   | 55625        | 2588          | 0.2014         | 7            


### What did the sandbox actually do?

In [10]:
print_trace(npm_sandbox)

The model ran 7 sandbox cell(s).

CELL 1   (exit=1, 7314 ms)
import pplx_sdk, json
urls=['https://www.npmjs.com/package/react','https://www.npmjs.com/package/lodash']
res=pplx_sdk.content.fetch(urls, cache_enabled=False)
for r in res:
 print('URL',r.get('url'),'error',r.get('error'),'title',r.get('title'))
 c=r.get('content') or ''
 print(c[:2000].replace('\n',' ')[:2000])
 print('---')
--- stderr ---
Traceback (most recent call last):
  File "<stdin>", line 5, in <module>
AttributeError: 'pplx_sdk.PageResult' object has no attribute 'get'


CELL 2   (exit=1, 3259 ms)
import pplx_sdk, json
urls=['https://www.npmjs.com/package/react','https://www.npmjs.com/package/lodash']
res=pplx_sdk.content.fetch(urls, cache_enabled=False)
for rr in res:
 r=dict(rr)
 print('URL',r.get('url'),'error',r.get('error'),'title',r.get('title'))
 c=r.get('content') or ''
 print(c[:3000].replace('\n',' ')[:3000])
 print('---')
--- stderr ---
Traceback (most recent call last):
  File "<stdin>", line 5, in <mod

## Example 2: Usain Bolt records: `web_search` vs `sandbox`

**Question:** four facts about Usain Bolt's world records: his 100 m and 200 m record times, and the
city and year he set them.

These are quoted in countless articles, so this is the opposite case: `web_search` finds them right in
the snippets. It's a fair check on when you **don't** need the sandbox.

In [ ]:
BOLT_PROMPT = (
    "I need four facts about Usain Bolt's athletics world records, each with a source URL: "
    "(1) his 100 m world-record time in seconds; (2) his 200 m world-record time in seconds; "
    "(3) the city where he set both; (4) the year he set both."
)

BOLT_KEY = {"100m": "9.58", "200m": "19.19", "city": "Berlin", "year": "2009"}

def grade_bolt(resp):
    text = answer_text(resp)
    checks = [v in text for v in BOLT_KEY.values()]
    return sum(checks), len(checks)

### Run both ways

In [11]:
bolt_baseline = run(BOLT_PROMPT, use_sandbox=False)
bolt_sandbox  = run(BOLT_PROMPT, use_sandbox=True)

print("\nsandbox answer:\n", answer_text(bolt_sandbox))

Submitting [web_search only] …
  resp_dedce1cd-af30… -> queued
  resp_dedce1cd-af30… -> in_progress
  resp_dedce1cd-af30… -> completed
Submitting [web_search + sandbox] …
  resp_90f8cf97-9c7a… -> queued
  resp_90f8cf97-9c7a… -> in_progress
  resp_90f8cf97-9c7a… -> in_progress
  resp_90f8cf97-9c7a… -> in_progress
  resp_90f8cf97-9c7a… -> in_progress
  resp_90f8cf97-9c7a… -> completed

sandbox answer:
 | # | Fact requested | Answer | Source URL |
|---|---|---:|---|
| 1 | Usain Bolt’s 100 m world-record time | **9.58 seconds** | https://worldathletics.org/records/by-discipline/sprints/100-metres/all/men |
| 2 | Usain Bolt’s 200 m world-record time | **19.19 seconds** | https://worldathletics.org/records/by-discipline/sprints/200-metres/outdoor/men |
| 3 | City where he set both records | **Berlin, Germany** | 100 m: https://worldathletics.org/records/by-discipline/sprints/100-metres/all/men ; 200 m: https://worldathletics.org/records/by-discipline/sprints/200-metres/outdoor/men |
| 4 | Ye

### Scorecard

In [12]:
print_scorecard([
    make_row("web_search only",      bolt_baseline, *grade_bolt(bolt_baseline)),
    make_row("web_search + sandbox", bolt_sandbox,  *grade_bolt(bolt_sandbox)),
])

run                  | correct | input_tokens | output_tokens | total_cost_usd | sandbox_cells
---------------------+---------+--------------+---------------+----------------+--------------
web_search only      | 4/4     | 16362        | 917           | 0.10204        | 0            
web_search + sandbox | 4/4     | 18378        | 1144          | 0.08359        | 3            


Both runs land all four facts at about the same cost: on a question the snippets already answer, the
sandbox buys you nothing. Reach for plain `web_search` here -- it's the simpler tool. (The sandbox
answer is still fully auditable in the trace below.)

### The sandbox's trace

Even here it really searches the web -- `pplx_sdk.search.web` plus the usual `WebHit` self-correction -- it just isn't worth the extra steps when the snippet already had the answer.

In [13]:
print_trace(bolt_sandbox)

The model ran 3 sandbox cell(s).

CELL 1   (exit=1, 3945 ms)
import pplx_sdk
queries=["Usain Bolt 100m world record 9.58 Berlin 2009 World Athletics", "Usain Bolt 200m world record 19.19 Berlin 2009 World Athletics"]
for q in queries:
    print('QUERY',q)
    hits=pplx_sdk.search.web(q, limit=5, domains=['worldathletics.org'])
    for h in hits:
        print(dict(h))
    print()

--- stdout ---
QUERY Usain Bolt 100m world record 9.58 Berlin 2009 World Athletics

--- stderr ---
Traceback (most recent call last):
  File "<stdin>", line 7, in <module>
TypeError: 'pplx_sdk.WebHit' object is not iterable


CELL 2   (exit=0, 3781 ms)
import pplx_sdk, json
hits=pplx_sdk.search.web("Usain Bolt 100m world record 9.58 Berlin 2009 World Athletics", limit=3, domains=['worldathletics.org'])
for h in hits:
    print(type(h))
    print(dir(h)[:30])
    print(getattr(h,'__dict__',None))
    print(h)

--- stdout ---
<class 'pplx_sdk.WebHit'>
['__class__', '__copy__', '__deepcopy__', '__delattr__', '__

## Takeaways

1. **Use `web_search` when the answer is written down.** For well-documented facts the sandbox adds no
   accuracy and no real cost saving, so the simpler tool wins -- Example 2.
2. **Reach for `sandbox` the moment it isn't.** Both tools search the same web, but the sandbox does it
   as code — search, fetch, parse — so the bulky page content never lands in the model's context. When
   a fact lives in metadata or a long page, that's cheaper *and* more accurate — Example 1.
3. **The sandbox answer is auditable.** Every value traces back to a page it searched for and read,
   right there in the trace.

**Try next:** swap in your own snippet-hostile question -- a number in a PDF table, a clause in a
filing, a value behind a lookup -- and watch the two columns separate.

> [Perplexity Research — *Rethinking Search as Code Generation*](https://research.perplexity.ai/articles/rethinking-search-as-code-generation)